# SSTADEX - Structured and Systematic Analog Design Exploration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lild4d4/sscs-ose-code-a-chip.github.io/blob/main/ISSCC26/submitted_notebooks/SSTADEX/SSTADEX.ipynb)
[![License: Apache 2.0](https://img.shields.io/badge/License-Apache_2.0-blue.svg)](https://www.apache.org/licenses/LICENSE-2.0)
</br>

#### Team Members 

|Name|Affiliation|IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|
| Daniel Arevalos (PhD Student) <br /> Email ID: daniel.arevalos@hm.edu|Hochschule Munchen| Yes |No|
| Dr. Stefan Wallentowitz (Profesor Advisor) <br /> Email ID: stefan.wallentowitz@hm.edu|Hochschule Munchen| Yes |Yes|

## Abstract

This notebook presents SSTADEX, a structured and systematic flow for analog design exploration based on symbolic analysis, lookup-table generation, and hierarchical circuit evaluation. The objective is to show how device-level characterization can be transformed into reusable design knowledge and then composed into higher-level analog blocks with increasing architectural complexity. Compared with our earlier work ([MDPI Electronics 14(17):3448](https://doi.org/10.3390/electronics14173448)), this notebook reflects a more solid and practical implementation of the flow, with clearer hierarchy handling, result propagation, condition filtering, and support for reusable primitives and macromodels.

The material is organized as a progressive design journey. It starts from the generation and interpretation of transistor-level lookup tables, continues with the construction of reusable analog primitives, and then applies the methodology to two amplifier case studies: a one-stage OTA and a two-stage OTA. Through these examples, the notebook demonstrates how SSTADEX can reduce manual trial-and-error, make design assumptions explicit, and support systematic exploration across multiple abstraction levels while preserving consistency between circuit constraints, intermediate variables, and final design targets.


## 1. Introduction

Analog circuit design still relies heavily on designer intuition, iterative simulation, and manual tuning across a large multi-dimensional space of electrical, geometrical, and topological decisions. While this workflow can produce excellent results in expert hands, it is difficult to scale, hard to document rigorously, and often inefficient when design constraints interact across multiple hierarchical levels. These limitations become even more visible in educational, exploratory, and open-design settings, where transparency and reproducibility are as important as final performance.

SSTADEX addresses this challenge by combining symbolic small-signal reasoning with structured exploration over precomputed device data. Instead of treating each circuit sizing step as an isolated numerical search, the flow organizes the problem around reusable building blocks, explicit specifications, and progressively refined design spaces. Device behavior is first captured through lookup tables, then abstracted into primitives, and finally composed into larger analog subsystems. This creates a bridge between low-level transistor characterization and high-level architectural reasoning, enabling a more systematic path from technology data to circuit-level decisions.

This notebook presents that methodology through two case studies: the design of a one-stage OTA and a two-stage OTA. The objective is to show how SSTADEX supports a more transparent and systematic path from device data to circuit-level decisions, while preserving consistency between low-level characterization, intermediate design constraints, and final performance targets.


### 1.1 Background

SSTADEX combines two complementary perspectives: structured analog design and systematic design. In structured analog design, complex circuits are decomposed into a hierarchy of functional blocks rather than treated as monolithic transistor-level systems. This improves modularity, reuse, and interpretability. Systematic design complements this view by replacing purely manual tuning with explicit exploration and filtering procedures driven by device data and circuit specifications.

In SSTADEX, this combination is realized through LUT-based primitive characterization and symbolic macromodel-based abstraction. Primitive blocks provide technology-aware electrical behavior, while macromodels enable higher-level exploration, specification evaluation, and hierarchical refinement. Together, these elements support an efficient design workflow that remains closely tied to the electrical meaning of the circuit.

> For LUT generation, this work reuses and adapts code from Mosplot: The MOSFET Characterization Tool by Mohamed Watfa (GitHub repository, March 2025, accessed March 31, 2025: https://github.com/medwatt/gmid). In this notebook, that characterization layer is used as the starting point for device-data generation, while SSTADEX builds on top of it to enable primitive abstraction, symbolic exploration, and hierarchical analog design.

### 1.2 Notebook Overview

This notebook is organized as a progressive walkthrough of the SSTADEX workflow, moving from conceptual foundations to practical design examples of increasing complexity. It begins with the setup of the required tools and the installation of SSTADEX, followed by an introduction to the basic concepts of the flow through LUT generation and simple device-level visualizations.

The next sections focus on primitive construction and characterization, showing how reusable analog building blocks are implemented and explored within the framework. Once these foundations are established, the methodology is demonstrated through three case studies: the design of a one-stage OTA, the design of a two-stage OTA, and finally the design of an LDO. The notebook concludes with a summary of the main insights and lessons obtained from these examples.

![alt text](figures/sstadex_flow.png "SSTADEX")

### 1.3 Workflow at a Glance

At a high level, the SSTADEX workflow follows a hierarchical and specification-driven design process. The flow starts from transistor-level characterization and progressively moves toward higher abstraction levels, where primitives are assembled into macromodels and evaluated against circuit-level requirements.

The workflow can be summarized in the following steps:

1. Technology characterization through LUT generation.
2. Primitive definition and characterization.
3. Macromodel construction.
4. Specification and testbench definition.
5. Hierarchical exploration and filtering.
6. Parameter update and refinement across the hierarchy.
7. Validation through complete case studies.

The following sections implement these steps progressively, beginning with the preparation of the environment and continuing through primitives, hierarchical circuit design, and final validation.

## 2. Environment Setup and Tool Installation

This section prepares the execution environment required for the examples in this notebook. The setup includes the installation of system dependencies, creation of a Python environment, retrieval of the SSTADEX repository, installation of the required Python packages, and configuration of the IHP SG13G2 PDK together with ngspice support.

### 2.1 System Dependencies and Python Environment

The following commands install the system-level dependencies required by the notebook and create an isolated Python environment using `micromamba`.


In [1]:
import os
import pathlib
import sys

# Install system packages required for compilation and simulation
!apt update -y
!apt install -y time build-essential bison flex libx11-dev libreadline-dev libfftw3-dev libxcursor-dev libxaw7-dev libtool autoconf automake curl

# Install micromamba
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create isolated environment
conda_prefix_path = pathlib.Path("conda-env")
CONDA_PREFIX = str(conda_prefix_path.resolve())
%env CONDA_PREFIX={CONDA_PREFIX}

!mkdir -p {CONDA_PREFIX}/conda-meta
!echo 'python ==3.13*' >> {CONDA_PREFIX}/conda-meta/pinned

!bin/micromamba create --yes --prefix $CONDA_PREFIX
!bin/micromamba install --yes --prefix $CONDA_PREFIX --channel conda-forge python=3.13 pip svgutils

# Expose environment paths to the notebook session
site_package_path = conda_prefix_path / "lib/python3.13/site-packages"
sys.path.append(str(site_package_path.resolve()))

PATH = os.environ["PATH"]
%env PATH={PATH}:{CONDA_PREFIX}/bin

LD_LIBRARY_PATH = os.environ.get("LD_LIBRARY_PATH", "")
%env LD_LIBRARY_PATH={LD_LIBRARY_PATH}:{CONDA_PREFIX}/lib

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.7 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]3m
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,968 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   m
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]  
Get:13 http://securit

### 2.2 Repository Setup and Python Package Installation

This step retrieves the SSTADEX repository and installs the Python packages required by the framework and its supporting modules.

In [2]:
# Clone the repository
!git clone --branch main https://github.com/lild4d4/SSTADEx.git --recursive

# Checkout the exact validated commit
%cd SSTADEx
!git checkout b5bef194c6bed3f6f4ec5cabe5faa91ad25c7e5b
!git submodule update --init --recursive
%cd ..

# Install Python packages from the repository
!$CONDA_PREFIX/bin/pip install ./SSTADEx
!$CONDA_PREFIX/bin/pip install ./SSTADEx/gmid


Cloning into 'SSTADEx'...
remote: Enumerating objects: 1381, done.
remote: Counting objects: 100% (487/487), done.
remote: Compressing objects: 100% (307/307), done.
remote: Total 1381 (delta 277), reused 353 (delta 156), pack-reused 894 (from 2)
Receiving objects: 100% (1381/1381), 127.09 MiB | 24.10 MiB/s, done.
Resolving deltas: 100% (712/712), done.
Updating files: 100% (380/380), done.
Submodule 'IHP-Open-PDK' (https://github.com/IHP-GmbH/IHP-Open-PDK.git) registered for path 'IHP-Open-PDK'
Submodule 'gmid' (https://github.com/lild4d4/gmid.git) registered for path 'gmid'
Cloning into '/content/SSTADEx/IHP-Open-PDK'...
remote: Enumerating objects: 20037, done.        
remote: Counting objects: 100% (3042/3042), done.        
remote: Compressing objects: 100% (407/407), done.        
remote: Total 20037 (delta 2862), reused 2635 (delta 2635), pack-reused 16995 (from 2)        
Receiving objects: 100% (20037/20037), 414.24 MiB | 20.83 MiB/s, done.
Resolving deltas: 100% (13128/13128)

### 2.3 OpenVAF, PDK, and ngspice Configuration

The final setup step configures the PDK, installs the OpenVAF binary, compiles ngspice support for the selected process (may take a while sorry but needed the osdi support), and links the required `.spiceinit` file.

In [ ]:
# Set PDK-related environment variables
%env PDK_ROOT=/content/SSTADEx/IHP-Open-PDK
%env PDK=ihp-sg13g2

PDK_ROOT = os.environ["PDK_ROOT"]
PDK = os.environ["PDK"]

# Install OpenVAF
!mkdir -p bin
!curl -Ls https://openva.fra1.cdn.digitaloceanspaces.com/openvaf_23_5_0_linux_amd64.tar.gz | tar -xzv -C bin
!chmod +x ./bin/openvaf

os.environ["PATH"] = os.environ["PATH"] + ":" + os.path.abspath("./bin")
!which openvaf

# Build ngspice
!git clone git://git.code.sf.net/p/ngspice/ngspice
!cd ngspice && ./compile_linux.sh

# Install ngspice support from the selected PDK
!python3 SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/install.py

# Link local spice initialization file
!ln -sf /content/SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/.spiceinit ~/.spiceinit
!ls -l ~/.spiceinit


env: PDK_ROOT=/content/SSTADEx/IHP-Open-PDK
env: PDK=ihp-sg13g2
openvaf
/content/bin/openvaf
Cloning into 'ngspice'...
remote: Enumerating objects: 140022, done.
remote: Counting objects: 100% (140022/140022), done.
remote: Compressing objects: 100% (27541/27541), done.
remote: Total 140022 (delta 113860), reused 136531 (delta 111124), pack-reused 0 (from 0)
Receiving objects: 100% (140022/140022), 45.95 MiB | 17.54 MiB/s, done.
Resolving deltas: 100% (113860/113860), done.
Removing files to be remade
Running libtoolize
libtoolize: putting auxiliary files in '.'.
libtoolize: copying file './ltmain.sh'
libtoolize: putting macros in AC_CONFIG_MACRO_DIRS, 'm4'.
libtoolize: copying file 'm4/libtool.m4'
libtoolize: copying file 'm4/ltoptions.m4'
libtoolize: copying file 'm4/ltsugar.m4'
libtoolize: copying file 'm4/ltversion.m4'
libtoolize: copying file 'm4/lt~obsolete.m4'
Running aclocal 
Running autoheader
Running automake -Wall --copy --add-missing
configure.ac:46: installing './ar-lib'
c

## 3. SSTADEX Basics

This section covers the foundations of the SSTADEX workflow. It begins with LUT generation for transistor characterization, then moves to the definition of analog primitives, and finally shows how these primitives can be explored before being used in larger hierarchical designs.

### 3.1 LUTs generation

The workflow starts with device characterization, where LUTs are generated over the selected bias and geometry ranges.

In [4]:
from mosplot.lookup_table_generator.simulators import NgspiceSimulator
from mosplot.lookup_table_generator import TransistorSweep, LookupTableGenerator
import os
from pathlib import Path

PDK_ROOT=os.environ['PDK_ROOT']
PDK=os.environ['PDK']

ngspice_nmos = NgspiceSimulator(
    simulator_path="ngspice",       
    temperature=27,                
    lib_mappings = [(PDK_ROOT+'/'+PDK+"/libs.tech/ngspice/models/cornerMOSlv.lib", "mos_tt")],
    osdi_paths = [(PDK_ROOT+'/'+PDK+"/libs.tech/ngspice/osdi/psp103_nqs.osdi")],
    mos_spice_symbols = ("XM1", "n.xm1.nsg13_lv_nmos"),
    device_parameters = {"w": 5e-6,}
)
ngspice_pmos = NgspiceSimulator(
    simulator_path="ngspice",       
    temperature=27,                 
    lib_mappings = [(PDK_ROOT+'/'+PDK+"/libs.tech/ngspice/models/cornerMOSlv.lib", "mos_tt")],
    osdi_paths = [(PDK_ROOT+'/'+PDK+"/libs.tech/ngspice/osdi/psp103_nqs.osdi")],
    mos_spice_symbols = ("XM1", "n.xm1.nsg13_lv_pmos"),
    device_parameters = {"w": 5e-6,}
)

# Define a sweep object for NMOS transistors.
nmos_sweep = TransistorSweep(
    mos_type="sg13_lv_nmos",
    vgs=(0.1, 1.2, 0.01),
    vds=(0.1, 1.2, 0.01),
    vbs=(0, -1.0, -0.1),
    length=[0.4e-6, 0.8e-6, 1.6e-6, 3.2e-6, 6.4e-6]
)
# Define a sweep object for PMOS transistors.
pmos_sweep = TransistorSweep(
    mos_type="sg13_lv_pmos",
    vgs=(-0.1, -1.2, -0.01),
    vds=(-0.1, -1.2, -0.01),
    vbs=(0, 1.0, 0.1),
    length=[0.4e-6, 0.8e-6, 1.6e-6, 3.2e-6, 6.4e-6]
)

obj_nmos = LookupTableGenerator(
    description="OpenPDK ihp-sg13g2",
    simulator=ngspice_nmos,
    model_sweeps={"sg13_lv_nmos": nmos_sweep},
    n_process=1,
)
obj_pmos = LookupTableGenerator(
    description="OpenPDK ihp-sg13g2",
    simulator=ngspice_pmos,
    model_sweeps={"sg13_lv_pmos": pmos_sweep},
    n_process=1,
)

SSTADEX = Path("./SSTADEx").resolve()
obj_nmos.build(str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_nmos"))
obj_pmos.build(str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_pmos"))


### 3.2 Primitives Definition

Once the device data is available, the next step is to encapsulate relevant behavior into reusable analog primitives.

As a basic primitive example, this notebook uses a simple differential pair. In SSTADEX, the primitive is defined from a LUT-driven transistor characterization together with an explicit port description, allowing the block to be evaluated as a reusable analog building element rather than as an isolated transistor-level script.

The simplediffpair primitive represents a symmetric differential input stage with ports VINP, VINN, VOUTP, VOUTN, and VTAIL. Its internal construction assumes that the tail current is split equally between the two branches, and for each operating point it derives the effective transistor conditions from the terminal voltages, particularly through the corresponding VGS and VDS values used to query the lookup table.

The following example shows how a simple differential pair can be instantiated, biased through its ports, and converted into a dataframe of reusable small-signal and sizing-related quantities.

In [104]:
import numpy as np
import pandas as pd
from pathlib import Path
from sympy import Symbol

from sstadex.models import Library

# Load the analog primitive library
analoglib_root = Path("SSTADEx/analoglib/primitives")
lib = Library(
    name="ihp_sg13g2",
    lut_files={
        "nmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_nmos.npz"),
        "pmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_pmos.npz"),
    })
lib.register_all(analoglib_root)

# Basic operating conditions
tail_current = 20e-6
input_common_mode = 0.9
output = 1
tail_voltage_sweep = np.linspace(0.2, 1, 5)

# Instantiate the primitive
diffpair = lib.get("simplediffpair", il=tail_current)

# Define port voltages
diffpair.set_port_voltages({
    "VINP": input_common_mode,
    "VINN": input_common_mode,
    "VOUTP": output,
    "VOUTN": output,
    "VTAIL": tail_voltage_sweep,
})
print(diffpair.summary())

# Build the primitive dataframe from LUT-based characterization
diffpair_df = diffpair.build()

# Map the results into SSTADEX parameters and outputs
diffpair.parameters = {
    Symbol("g_gm_xdp_m1"): diffpair_df["gm"].values,
    Symbol("R_gds_xdp_m1"): diffpair_df["Ro"].values,
}

diffpair.outputs = {
    Symbol("W_diff"): diffpair_df["width"].values,
    Symbol("L_diff"): diffpair_df["length"].values,
}

# Preview results
display(diffpair_df)
print(f"Number of explored operating points: {len(diffpair_df)}")


### 3.3 Visualization

Before moving to larger hierarchical explorations, SSTADEX also allows quick visualization of primitive behavior. This is useful to verify basic trends and identify feasible operating regions directly at the primitive level.

In [105]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(
    ax=axs[0],
    data=diffpair_df,
    x="gm",
    y="Ro",
    hue="length",
    palette="flare",
)
axs[0].set_xscale("log")
axs[0].set_yscale("log")
axs[0].set_title("Transconductance vs Output Resistance")
axs[0].set_xlabel("gm [A/V]")
axs[0].set_ylabel("Ro [Ohm]")

sns.scatterplot(
    ax=axs[1],
    data=diffpair_df,
    x="width",
    y="gm",
    hue="length",
    palette="viridis",
)
axs[1].set_xscale("log")
axs[1].set_yscale("log")
axs[1].set_title("Required Width vs Transconductance")
axs[1].set_xlabel("Width [m]")
axs[1].set_ylabel("gm [A/V]")

plt.tight_layout()
plt.show()

## 4. One-Stage OTA Design

This section presents the first complete circuit-level example in the notebook: a one-stage OTA built from reusable SSTADEX primitives. The goal is to show how primitive-level characterization can be translated into a hierarchical macromodel, explored under specification constraints, and then validated through circuit simulation.

The following figures summarize the circuit topology and its hierarchical decomposition into reusable blocks:

![alt text](figures/One-Stage-Diagram.png "Diagram")
![alt text](figures/One-Stage-Hierarchy.png "Hierarchy")

### 4.1 Imports

The following imports provide the numerical, symbolic, plotting, and SSTADEX utilities required for the example.

In [106]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np
from sympy import Symbol
from sympy import lambdify
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
sns.color_palette("mako")

from sstadex import Macromodel, Test, dfs, Testbench, VoltageSource, CurrentSource, Resistor, Capacitor

SSTADEX = Path("./SSTADEx").resolve()
if str(SSTADEX) not in sys.path:
    sys.path.insert(0, str(SSTADEX))

from sstadex.models import Library

N_points = 10
lib = Library(
    name      = "ihp_sg13g2",
    lut_files = {
        "nmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_nmos.npz"),
        "pmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_pmos.npz"),
    },
)

lib.register_all(SSTADEX / "analoglib/primitives/")
print(lib)

### 4.2 Specifications Definition

The one-stage OTA example starts by defining the electrical operating conditions, design targets, and the parameter ranges used during exploration.

In [107]:
## Electrical parameters
Vout = 1
Vref = 0.9                                       
Vdd = 1.5                           
CL = 1e-12 

## OTA specification
gain_condition = 30

I_bias_ref = 20e-6
I_amp = 20e-6 

lengths = [4e-7, 8e-7, 1.6e-6, 3.2e-6, 6.4e-6]

### 4.3 Primitives Definition

The OTA is constructed from three primitive blocks: a differential pair, a current mirror used as active load, and a current source used for bias generation. Each primitive is instantiated from the analog library, biased through its ports, and then characterized into reusable parameters and sizing outputs.

In [7]:
vs = np.linspace(0.2, Vout-0.1, N_points)

diffpair = lib.get("simplediffpair", il=I_amp)
currentmirror = lib.get("simplecurrentmirror", il=I_amp)
currentsource = lib.get('simplecurrentsource', il=I_amp)

diffpair.set_port_voltages({"VINP": Vref, "VINN": Vref, "VOUTP": Vout, "VOUTN": Vout, "VTAIL": vs})
currentmirror.set_port_voltages({"VINP": Vout, "VINN": Vout, "VOUTP": Vout, "VOUTN": Vout, "VDD": Vdd})
currentsource.set_port_voltages({"VOUTP": vs, "VSS": 0, "VINP": vs})

print(diffpair.summary())
print(currentmirror.summary())
print(currentsource.summary())

diffpair_df = diffpair.build()
currentmirror_df = currentmirror.build()
currentsource_df = currentsource.build(ref_current=I_amp)

diffpair_df.to_csv('diffpair.csv')
currentmirror_df.to_csv('currentmirror.csv')
currentsource_df.to_csv('currentsource.csv')

diffpair.parameters = {Symbol('g_gm_xdp_m1'): diffpair_df['gm'].values, Symbol('R_gds_xdp_m1'): diffpair_df['Ro'].values}
currentmirror.parameters = {Symbol('g_gm_xcm_m1'): currentmirror_df['gm'].values, Symbol('R_gds_xcm_m1'): currentmirror_df['Ro'].values}
currentsource.parameters = {Symbol('g_gm_cs_m1'): currentsource_df['gm'].values, Symbol('R_gds_cs_m1'): currentsource_df['Ro'].values}

diffpair.outputs = {Symbol("W_diff"): diffpair_df["width"].values, Symbol("L_diff"): diffpair_df["length"].values}
currentmirror.outputs = {Symbol("W_al"): currentmirror_df["width"].values, Symbol("L_al"): currentmirror_df["length"].values}
currentsource.outputs = {Symbol("W_cs_m1"): currentsource_df["width_m1"].values, Symbol("W_cs_m2"): currentsource_df["width_m2"].values, Symbol("L_cs"): currentsource_df["length"].values}

### 4.4 Macromodels Definition

Once the primitive blocks are defined, they are assembled into higher-level macromodels. The main OTA macromodel captures the first-stage amplifier behavior, while the bias branch is represented as a dedicated submacromodel so that it can be explored, filtered, and updated hierarchically.

In [ ]:
OTA_1stage_macro = Macromodel(
    name = 'OTA_1stage_macro',
    ports=["VINP", "VINN", "VOUT", "VDD", "IBIAS", "Vbias", "VSS"],
    electrical_parameters = {"Vdd": Vdd, "Vneg": Vref, "Vout": Vout, "Il": I_amp},
    outputs = [Symbol("W_diff"), Symbol("L_diff"),Symbol("W_al"), Symbol("L_al")],
    macromodel_parameters={Symbol('Ra'): np.logspace(3, 7, N_points), Symbol('gma'): np.logspace(-5, -2, N_points)}
    )

current_source_macro = Macromodel(
    name = 'current_source_macro',
    ports = ['VOUT', 'VSS', 'Vbias'],
    electrical_parameters = {'Iref': I_bias_ref, 'Ibias': I_amp, 'Vdd': Vdd},
    outputs = [Symbol("W_cs_m1"), Symbol("W_cs_m2"), Symbol("L_cs"),
    ],
    model="""I{INSTANCE} {VOUT} {VSS} 0""",
    macromodel_parameters = {Symbol('Ixcs_macro'): [I_amp]}
)

### 4.5 Building the Hierarchy: Interface variables, Shared Nodes and Instances

This step defines how information flows across the hierarchy. Interface variables are used to propagate internal operating-point quantities through the dataframes, shared nodes enforce consistency between blocks that must satisfy the same internal voltage, and instances define how primitives and submacromodels are connected inside the OTA hierarchy.

In [ ]:

diffpair.interface_variables={'vs_diff': np.tile(vs, len(lengths))}
currentsource.interface_variables={'vs_cs': np.tile(vs, len(lengths)),}

OTA_1stage_macro.interface_variables = ["vs_diff"]
current_source_macro.interface_variables=["vs_cs"]

OTA_1stage_macro.shared_nodes = {"IBIAS_node": ["vs_diff", "vs_cs"]}

In [ ]:
OTA_1stage_macro.add_instance(
    "xdp",
    diffpair,
    {"VINP": "VINP", "VINN": "VINN", "VOUTP": "VOUT", "VOUTN": "N1", "VTAIL": "IBIAS"},
    index=0,
    netlist_params={"W_diff": Symbol("W_diff"), "L_diff": Symbol("L_diff"), "ng_diff": Symbol("ng_diff")},
)

OTA_1stage_macro.add_instance(
    "xcm",
    currentmirror,
    {"VINP": "N1", "VINN": "N1", "VOUTP": "VOUT", "VOUTN": "N1", "VDD": "VDD"},
    index=0,
    netlist_params={"W_al": Symbol("W_al"), "L_al": Symbol("L_al"), "ng_al": Symbol("ng_al")},
)

OTA_1stage_macro.add_instance(
    "xcs_macro",
    current_source_macro,
    {"VOUT": "IBIAS", "VSS": "VSS", "Vbias": "Vbias"},
    index=0,
    netlist_params={'Ibias': Symbol('Ibias')}
)

current_source_macro.add_instance(
    "xcs",
    currentsource,
    {"VOUTP": "VOUT", "VSS": "VSS", "VOUTN": "Vbias", "VINP": "Vbias", "VINN": "Vbias"},
    index=0,
    netlist_params={"W_cs_m1": Symbol("W_cs_m1"), "W_cs_m2": Symbol("W_cs_m2"), "L_cs": Symbol("L_cs"), "ng_cs_m1": Symbol("ng_cs_m1"), "ng_cs_m2": Symbol("ng_cs_m2")}
)

### 4.6 Conditions and Propagation Definition

In addition to circuit-level specifications, local sizing constraints can also be applied to primitives and submacromodels. These propagated conditions reduce the exploration space and ensure that only physically meaningful design points are preserved during the hierarchical flow.

In [ ]:
OTA_1stage_macro.propagated_conditions = {
    "direct": [
        {
            "kind": "range",
            "column": Symbol("W_al"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
        {
            "kind": "range",
            "column": Symbol("W_diff"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
    ],
    "derived": [],
}

current_source_macro.propagated_conditions = {
    "direct": [
        {
            "kind": "range",
            "column": Symbol("W_cs_m1"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
        {
            "kind": "range",
            "column": Symbol("W_cs_m2"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
    ],
    "derived": [],
}

### 4.7 Testbenches

The OTA metrics are evaluated through symbolic testbenches. In this example, gain and output resistance are evaluated directly, while the effective transconductance is derived as a composed quantity from the previously evaluated results.

In [12]:
tb_gain = Testbench(
    name="ota_1stage_gain",
    dut=OTA_1stage_macro,
    view="small_signal",
    elements=[
        VoltageSource("Vdd", "VDD", "VSS", 0),
        VoltageSource("Vss", "VSS", "VSS", 0),
        VoltageSource("V_n", "VINN", "VSS", 0),
        VoltageSource("V_p", "VINP", "VSS", 1),
    ],
    tf=("VOUT", "VINP"),
    parameter_map={
        Symbol("Vdd"): 0,
        Symbol("Vss"): 0,
        Symbol("V_n"): 0,
        Symbol("V_p"): 1,
        Symbol("g_gm_xdp_m2"): Symbol("g_gm_xdp_m1"),
        Symbol("R_gds_xdp_m2"): Symbol("R_gds_xdp_m1"),
        Symbol("g_gm_xcm_m2"): Symbol("g_gm_xcm_m1"),
        Symbol("R_gds_xcm_m2"): Symbol("R_gds_xcm_m1"),
        Symbol("s"): 0,
    },
)

gain_1stage = tb_gain.make_test(
    name="gain_1stage",
    opt_goal="max",
    conditions={"min": [10 ** (-100 / 20)]},
)

tb_rout = Testbench(
    name="ota_1stage_rout",
    dut=OTA_1stage_macro,
    view="small_signal",
    elements=[
        VoltageSource("Vdd", "VDD", "VSS", 0),
        VoltageSource("Vss", "VSS", "VSS", 0),
        VoltageSource("V_n", "VINN", "VSS", 0),
        VoltageSource("V_p", "VINP", "VSS", 0),
        VoltageSource("Vr", "VR", "VSS", 1),
        Resistor("Rr", "VR", "VOUT", 1000),
    ],
    tf=("VOUT", "VR"),
    parameter_map={
        Symbol("Vdd"): 0,
        Symbol("Vss"): 0,
        Symbol("V_n"): 0,
        Symbol("V_p"): 0,
        Symbol("Vr"): 1,
        Symbol("Rr"): 1000,
        Symbol("g_gm_xdp_m2"): Symbol("g_gm_xdp_m1"),
        Symbol("R_gds_xdp_m2"): Symbol("R_gds_xdp_m1"),
        Symbol("g_gm_xcm_m2"): Symbol("g_gm_xcm_m1"),
        Symbol("R_gds_xcm_m2"): Symbol("R_gds_xcm_m1"),
        Symbol("s"): 0,
    },
)

rout_1stage = tb_rout.make_test(
    name="rout_1stage",
    opt_goal="max",
    target_param=Symbol("Ra"),
    conditions={"min": [1]},
    lamd=lambda x: x * 1000 / (1 - x),
)

tb_gm_1stage = Testbench(
    name="ota_1stage_gm",
    dut=OTA_1stage_macro,
    tf=("VOUT", "VINP"),
)

gm_1stage = tb_gm_1stage.make_test(
    name="gm_1stage",
    opt_goal="max",
    target_param=Symbol("ga"),
    composed=1,
    out_def = {"divide": [gain_1stage, rout_1stage]},
    conditions={"min": [1e-10]},
)

tb_ibias = Testbench(
    name="currentsource_ibas",
    dut=current_source_macro,
    view="small_signal",
    elements=[
        VoltageSource("Vdd", "VDD", "VSS", 0),
        VoltageSource("Vss", "VSS", "VSS", 0),
        CurrentSource("I0", "Vbias", "VSS", 0),
    ],
    tf=("VOUT", "Vbias"),
    parameter_map={
        Symbol("Vdd"): 0,
        Symbol("Vss"): 0,
        Symbol("I0"): 0,
        Symbol("s"): 0,
        Symbol('g_gm_cs_m2'): Symbol('g_gm_cs_m1'),
        Symbol('R_gds_cs_m2'): Symbol('R_gds_cs_m1')
    }
)

ibias_currentsource = tb_ibias.make_test(
    name="ibias_currentsource",
    opt_goal="max",
    conditions={"min": [0]},
)

### 4.8 Building

With the hierarchy and testbenches in place, the OTA can be explored using the SSTADEX flow. The first run evaluates the full feasible design space, while the second run applies Pareto filtering to keep only the most relevant tradeoff points.

>Readers are encouraged to inspect the log messages marked with [FLOW] during execution, as they provide a concise trace of how conditions are propagated and how the exploration space is progressively filtered through the hierarchy.

In [13]:
from pathlib import Path

osdi_dir = Path("output")
osdi_dir.mkdir(parents=True, exist_ok=True)

current_source_macro.specifications=[ibias_currentsource]
current_source_macro.primitives = [currentsource]
current_source_macro.num_level_exp = 1
current_source_macro.run_pareto = False

OTA_1stage_macro.specifications = [gain_1stage, rout_1stage, gm_1stage]
OTA_1stage_macro.opt_specifications = [gain_1stage]
OTA_1stage_macro.primitives = [diffpair, currentmirror]
OTA_1stage_macro.submacromodels = [current_source_macro]
OTA_1stage_macro.num_level_exp = -1
OTA_1stage_macro.is_primitive = 0
OTA_1stage_macro.run_pareto = False

_, _, _, ota_1stage_df, mask = dfs(OTA_1stage_macro, debug = False)

ota_1stage_df.to_csv('ota_1stage_df.csv')

On the previous exploration the parameter ```run_pareto``` is set to false, lets try with the pareto frontier !

In [ ]:
OTA_1stage_macro.run_pareto = True
_, _, _, ota_1stage_df_pareto, mask = dfs(OTA_1stage_macro, debug = False)
ota_1stage_df_pareto.to_csv('ota_1stage_df_pareto.csv')

### 4.9 Exploration Results

The resulting dataframes allow the explored design space to be inspected in terms of gain, bandwidth, transconductance, and area. The Pareto-filtered subset highlights the most efficient tradeoff points among the explored solutions.

In [ ]:
ota_1stage_df["gain"] = 20*np.log10(ota_1stage_df["gain_1stage"])
ota_1stage_df_pareto["gain"] = 20*np.log10(ota_1stage_df_pareto["gain_1stage"])
ota_1stage_df["bw"] = 1/(2*np.pi*ota_1stage_df["rout_1stage"]*CL)
ota_1stage_df_pareto["bw"] = 1/(2*np.pi*ota_1stage_df_pareto["rout_1stage"]*CL)

In [17]:
fig, axs = plt.subplots(2, 2, figsize=(15, 12))
sns.scatterplot(ax=axs[0,0], data=ota_1stage_df, x="area", y="gain", hue=Symbol("L_diff"), palette='flare')
sns.scatterplot(ax=axs[0,0], data=ota_1stage_df_pareto, x='area', y='gain', c='green')
sns.scatterplot(ax=axs[0,1], data=ota_1stage_df, x="area", y="bw", palette='flare')
sns.scatterplot(ax=axs[0,1], data=ota_1stage_df_pareto, x='area', y='bw', c='green')
sns.scatterplot(ax=axs[1,0], data=ota_1stage_df, x="area", y="gm_1stage", palette='flare')
sns.scatterplot(ax=axs[1,0], data=ota_1stage_df_pareto, x='area', y='gm_1stage', c='green')
sns.scatterplot(ax=axs[1,1], data=ota_1stage_df, x=Symbol("R_gds_xdp_m1"), y=Symbol("R_gds_xcm_m1"), hue="rout_1stage", palette='flare')
sns.scatterplot(ax=axs[1,1], data=ota_1stage_df_pareto, x=Symbol("R_gds_xdp_m1"), y=Symbol("R_gds_xcm_m1"), c='green')

axs[0,0].set_xscale('log')

axs[0,1].set_xscale('log')
axs[0,1].set_yscale('log')

axs[1,0].set_xscale('log')
axs[1,0].set_yscale('log')

These plots show how the one-stage OTA design space can be narrowed systematically before moving to detailed device-level verification. It also show the viability to use it as already defined submacromodel for another bigger design... We do that in the next exmaple !!

### 4.10 Verification

Finally, a subset of Pareto-selected solutions is translated back into circuit-level parameters and evaluated with ngspice. This step compares the symbolic exploration results against transistor-level simulation and helps assess the accuracy of the hierarchical model.

In [18]:
wmax = 5e-6

ng_diff = np.ceil(ota_1stage_df_pareto[Symbol("W_diff")].to_numpy() / wmax).astype(int)
ng_al = np.ceil(ota_1stage_df_pareto[Symbol("W_al")].to_numpy() / wmax).astype(int)
ng_cs_m1 = np.ceil(ota_1stage_df_pareto[Symbol("W_cs_m1")].to_numpy() / wmax).astype(int)
ng_cs_m2 = np.ceil(ota_1stage_df_pareto[Symbol("W_cs_m2")].to_numpy() / wmax).astype(int)

point = {
    Symbol("W_diff"): ota_1stage_df_pareto[Symbol("W_diff")].to_numpy(),
    Symbol("L_diff"): ota_1stage_df_pareto[Symbol("L_diff")].to_numpy(),
    Symbol("ng_diff"): ng_diff,
    Symbol("W_al"): ota_1stage_df_pareto[Symbol("W_al")].to_numpy(),
    Symbol("L_al"): ota_1stage_df_pareto[Symbol("L_al")].to_numpy(),
    Symbol("ng_al"): ng_al,
    Symbol("W_cs_m1"): ota_1stage_df_pareto[Symbol("W_cs_m1")].to_numpy(),
    Symbol("W_cs_m2"): ota_1stage_df_pareto[Symbol("W_cs_m2")].to_numpy(),
    Symbol("L_cs"): ota_1stage_df_pareto[Symbol("L_cs")].to_numpy(),
    Symbol("ng_cs_m1"): ng_cs_m1,
    Symbol("ng_cs_m2"): ng_cs_m2,
}

simulations = OTA_1stage_macro.ngspice_sim(
    point,
    variables=["gain", "vout", "ibias", "vbias"],
    extra_spice = {
        "pre": [
            "**",
            "x1 net1 vn vout vdd ibias vbias vss OTA_1stage_macro", # VINP VINN VOUTP VDD IBIAS
            "R1 vfb vout 100000000 m=1",
            "C2 vfb vss 10 m=1",
            "R2 vfb vss 900000000 m=1",
            "V1 net1 vfb dc 0 ac 1",
            "I0 vdd vbias 20e-6",
            ".lib /content/SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/models/cornerMOSlv.lib mos_tt",
            "Vref vn 0 0.9",
            "Vdd vdd 0 1.5",
            "Vss vss 0 0",
            ".control",
            "pre_osdi /content/SSTADEx/SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/osdi/psp103_nqs.osdi",
            "ac dec 10 1 1G",
            "meas ac gain find vdb(vout) at=1000",
            "op",
            "print v(vout)",
            "print v(ibias)",
            "print v(vbias)",
            ".endc",
            ".end"
        ]
    })

print(simulations)

sim_results_df = pd.DataFrame(
    [
        {
            **sim_result["params"],
            **{
                f"sim_{name}": value
                for name, value in sim_result.get("variables", {}).items()
            },
            "run_name": sim_result["run_name"],
            "returncode": sim_result["returncode"],
        }
        for sim_result in simulations
    ]
)

ota_1stage_df_pareto_comparison = pd.concat(
    [
        ota_1stage_df_pareto.reset_index(drop=True),
        sim_results_df[
            ["run_name", "returncode", "sim_gain", "sim_vout", "sim_ibias", "sim_vbias"]
        ].reset_index(drop=True),
    ],
    axis=1,
)

ota_1stage_df_pareto_comparison["gain_error"] = 100*np.abs(ota_1stage_df_pareto_comparison["gain"] - ota_1stage_df_pareto_comparison["sim_gain"])/ota_1stage_df_pareto_comparison["gain"]

ota_1stage_df_pareto_comparison["ibias_error"] = 100*np.abs(ota_1stage_df_pareto_comparison["vs_diff"] - ota_1stage_df_pareto_comparison["sim_ibias"])/ota_1stage_df_pareto_comparison["vs_diff"]

ota_1stage_df_pareto_comparison.to_csv("ota_1stage_comparison.csv", index=False)
ota_1stage_df_pareto_comparison

The comparison shows that the gain error remains minimal, while the bias-current error is still within an acceptable range for this level of abstraction. Further improvement is expected from increasing the accuracy and resolution of the underlying LUT characterization.

## 5. Two-Stage OTA Design

This section extends the previous example toward a two-stage OTA. A key aspect of the workflow is that the one-stage OTA developed in the previous section is encapsulated as a macromodel and added to the macromodel library, so that it can be reused here as a predefined hierarchical block without redefining its internal structure. In this way, the example also demonstrates how SSTADEX supports hierarchical reuse across design levels.

The following figures summarize the circuit topology of the two-stage OTA and its hierarchical decomposition into reusable blocks:

![alt text](figures/Two-Stage-Diagram.png "Diagram")
![alt text](figures/Two-Stage-Hierarchy.png "Hierarchy")

In [110]:
from sstadex import MacroLibrary

primitive_lib = Library(
    name="ihp_sg13g2",
    lut_files={
        "nmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_nmos.npz"),
        "pmos": str(SSTADEX / "LUTs/ihp-sg13g2/lv_5w_pmos.npz"),
    },
)
primitive_lib.register_all(SSTADEX / "analoglib/primitives")

macro_lib = MacroLibrary(
    name="ihp_sg13g2_macros",
    primitive_library=primitive_lib,
)
macro_lib.register_all(SSTADEX / "analoglib/macros")

## Electrical parameters

Vout = 1.1  
Vin = 1.5                             
Vref = 0.9                       
IL = 1e-3                                      
CL = 1e-12                                   
RL = Vout/IL

## OTA 2stage specifications

gain_condition = 70

I_bias_ref = 20e-6
I_amp_1stage = 20e-6 
I_amp_2stage = 20e-6 

lengths = [4e-7, 8e-7, 1.6e-6, 3.2e-6, 6.4e-6]

### 5.1 Reusing the One-Stage OTA Macromodel

The previously developed one-stage OTA is now imported from the macromodel library and used as a predefined first-stage block. This avoids redefining its internal structure and illustrates how SSTADEX supports hierarchical reuse across design levels.

In [111]:
N_points=10
vs = np.linspace(0.2, Vout-0.1, N_points)
vout_1stage = np.linspace(Vref - 0.1, Vout+0.1, N_points)

ota_1stage = macro_lib.get(
    "ota_1stage_macro",
    N_points=10, 
    ROOT=SSTADEX,
    lengths=lengths,
    electrical_parameters={
        "Vdd": Vin,
        "Vref": Vref,
        "Vout": vout_1stage,
        "I_amp": I_amp_1stage,
        "vs": vs
    },
    ports=["VINP", "VINN", "VOUT", "VBIAS", "VDD", "VSS"],
    model="""Ra {VOUT} {VDD} 1
Ga {VOUT} {VDD} {VINP} {VINN} 1"""
)


### 5.2 Second-Stage Primitive Blocks

The second stage is completed with additional primitive blocks, which are instantiated, biased, and characterized in the same way as in the previous example. These primitives provide the gain stage and biasing elements required to complete the two-stage architecture.

In [112]:
commonsource = primitive_lib.get("simplecommonsource", il=I_amp_2stage)
commonsource.set_port_voltages({"VIN": vout_1stage,"VOUT": Vout,"VDD": Vin})
commonsource_df = commonsource.build()
commonsource.parameters = {
    Symbol('g_gm_xcos_m1'): commonsource_df['gm'].values,
    Symbol('R_gds_xcos_m1'): commonsource_df['Ro'].values,
}
commonsource.outputs = {
    Symbol("W_cos"): commonsource_df["width"].values,
    Symbol("L_cos"): commonsource_df["length"].values,
}

currentsource = primitive_lib.get('simplecurrentsource', il=I_amp_2stage)
currentsource.set_port_voltages({"VOUTP": Vout, "VSS": 0, "VINP": Vout})
currentsource_df = currentsource.build(ref_current=I_bias_ref)
currentsource.parameters = {
    Symbol('g_gm_xcs_m1'): currentsource_df['gm'].values,
    Symbol('R_gds_xcs_m1'): currentsource_df['Ro'].values
}
currentsource.outputs = {
    Symbol("W_cs_2stage_m1"): currentsource_df["width_m1"].values,
    Symbol("W_cs_2stage_m2"): currentsource_df["width_m2"].values,
    Symbol("L_cs_2stage"): currentsource_df["length"].values,
}

### 5.3 Top-Level Macromodel Definition

Once the reusable first-stage macromodel and the additional primitives are available, they are assembled into a new top-level macromodel representing the complete two-stage OTA.



In [113]:
OTA_2stage_macro = Macromodel(
    name = 'OTA_2stage_macro',
    ports=["VINP", "VINN", "VOUT", "VBIAS_1STAGE", "VBIAS_2STAGE", "VDD", "VSS"],
    outputs = [
        Symbol("W_cos"), Symbol("L_cos"),
        Symbol("W_cs_2stage_m1"), Symbol("W_cs_2stage_m2"), Symbol("L_cs_2stage"),
        ],
    electrical_parameters = {
        "Vdd": Vin,
        "Vneg": Vref,
        "Vout": Vout,
        "Il": I_amp_2stage},
    macromodel_parameters={
        Symbol('Ra'): np.logspace(3, 7, N_points),
        Symbol('ga'): np.logspace(-5, -2, N_points),
    }
)

In [114]:
OTA_2stage_macro.add_instance(
    "xcos",
    commonsource,
    {
        "VIN": "VOUT_1STAGE",
        "VOUT": "VOUT",
        "VDD": "VDD",
    },
    index=0,
    netlist_params={
        "W_cos": Symbol("W_cos"),
        "L_cos": Symbol("L_cos"),
        "ng_cos": Symbol("ng_cos"),
    },
)

OTA_2stage_macro.add_instance(
    "xcs",
    currentsource,
    {
        "VOUTP": "VOUT",
        "VSS": "VSS", 
        "VOUTN": "VBIAS_2STAGE",
        "VINP": "VBIAS_2STAGE",
        "VINN": "VBIAS_2STAGE"
    },
    index=0,
    netlist_params={
        "W_cs_m1": Symbol("W_cs_2stage_m1"),
        "W_cs_m2": Symbol("W_cs_2stage_m2"),
        "L_cs": Symbol("L_cs_2stage"),
    }
)

OTA_2stage_macro.add_instance(
    'xota_1stage',
    ota_1stage,
    {
        "VINP": "VINP",
        "VINN": "VINN",
        "VOUT": "VOUT_1STAGE",
        "VBIAS": "VBIAS_1STAGE",
        "VDD": "VDD",
        "VSS": "VSS"
    },
    index=0,
)

### 5.5 Interface Variables and Shared Nodes

As in the one-stage example, interface variables are propagated through the hierarchy so that internal operating-point information remains visible during exploration. Shared-node constraints are then used to enforce consistency between blocks that must agree on the same internal node voltage.

In [115]:
commonsource.interface_variables={
    'vout_1stage_commonsource': np.repeat(vout_1stage, len(lengths))
}
OTA_2stage_macro.interface_variables=[
    "vout_1stage_commonsource"
]
OTA_2stage_macro.shared_nodes = {
"VOUT_node": ["vout_1stage_commonsource", "vout_1stage_diffpair"],
}

### 5.6 Condition Propagation to the Reused 

In this example, the parent OTA does not only reuse the one-stage macromodel structurally, but also constrains it functionally. A derived metric is defined for the first stage and used to propagate a gain-related condition from the two-stage design context back to the submacro, so that the first-stage exploration remains consistent with the requirements imposed by the full two-stage architecture.

In [116]:
ota_1stage.derived_metrics = {
    "gain_1stage_proxy": lambda df: df[Symbol("Ra")] * df[Symbol("ga")],
}

OTA_2stage_macro.submacro_condition_rules = {
    ota_1stage: [
        {
            "kind": "range_from_submacro_metric",
            "metric": "gain_1stage_proxy",
            "target_column": "gain_1stage",
            "bound": "min",
            "margin_factor": 1.0,
        },
    ]
}

OTA_2stage_macro.propagated_conditions = {
    "direct": [
        {
            "kind": "range",
            "column": Symbol("W_cos"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
    ],
    "derived": [],
}

ota_1stage.propagated_conditions = {
    "direct": [
        {
            "kind": "range",
            "column": Symbol("W_al"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
        {
            "kind": "range",
            "column": Symbol("W_diff"),
            "condition": {"min": 1e-6, "max": 1000e-6},
        },
    ],
    "derived": [],
}

### 5.7 Testbench and Hierarchical Exploration

In [117]:
tb_gain_2stage = Testbench(
    name="ota_2stage_gain",
    dut=OTA_2stage_macro,
    view="small_signal",
    elements=[
        VoltageSource("Vdd", "VDD", "VSS", 0),
        VoltageSource("Vss", "VSS", "VSS", 0),
        VoltageSource("V_n", "VINN", "VSS", 0),
        VoltageSource("V_p", "VINP", "VSS", 1),
    ],
    tf=("VOUT", "VINP"),
    parameter_map={
        Symbol("Vdd"): 0,
        Symbol("Vss"): 0,
        Symbol("V_n"): 0,
        Symbol("V_p"): 1,
        Symbol("s"): 0,
    },
)

gain_2stage = tb_gain_2stage.make_test(
    name="gain_2stage",
    opt_goal="max",
    conditions={"min": [10 ** (gain_condition / 20)]},
)

With the hierarchy and propagated conditions in place, the two-stage OTA can be explored as a complete macromodel. During execution, the [FLOW] log messages provide a compact trace of the hierarchical process, including propagated constraints, shared-node filtering, and dataframe size reductions across the exploration steps.

In [118]:
from pathlib import Path

output = Path("output")
output.mkdir(parents=True, exist_ok=True)

ota_1stage.num_level_exp = -1
ota_1stage.run_pareto = False

OTA_2stage_macro.specifications = [gain_2stage]
OTA_2stage_macro.opt_specifications = [gain_2stage]
OTA_2stage_macro.primitives = [commonsource, currentsource]
OTA_2stage_macro.submacromodels = [ota_1stage]
OTA_2stage_macro.num_level_exp = -1
OTA_2stage_macro.run_pareto = False

_, _, _, OTA_2stage_macro_df, mask = dfs(OTA_2stage_macro, debug = False)

Again lets run with the pareto frontier enable !

In [119]:
OTA_2stage_macro.run_pareto = True

_, _, _, OTA_2stage_macro_df_pareto, mask = dfs(OTA_2stage_macro, debug = False)

### 5.8 Exploration Results

The resulting dataframe captures the feasible design space of the two-stage OTA after hierarchical filtering and reuse of the first-stage macromodel. These results can then be visualized to inspect tradeoffs between area, total gain, and the internal contribution of the first stage.

In addition to the total two-stage gain, the dataframe also retains information associated with the reused first-stage block, making it possible to visualize how submacro-level performance contributes to the final system-level result.

In [121]:
OTA_2stage_macro_df["gain"] = 20*np.log10(OTA_2stage_macro_df["gain_2stage"])
OTA_2stage_macro_df_pareto["gain"] = 20*np.log10(OTA_2stage_macro_df_pareto["gain_2stage"])
OTA_2stage_macro_df["gain_1stage"] = 20*np.log10(OTA_2stage_macro_df[Symbol("Ra")]*OTA_2stage_macro_df[Symbol("ga")])

In [122]:
fig, ax = plt.subplots(figsize=(8, 6))

sns.scatterplot(
    ax=ax,
    data=OTA_2stage_macro_df,
    x="area",
    y="gain",
    hue="gain_1stage",
    palette="flare",
)

sns.scatterplot(
    ax=ax,
    data=OTA_2stage_macro_df_pareto,
    x="area",
    y="gain",
    color="green",
)

ax.set_xscale("log")
ax.set_title("Two-Stage OTA Exploration")
ax.set_xlabel("Area")
ax.set_ylabel("Gain [dB]")

plt.tight_layout()
plt.show()

### 5.9 Verification

This final step verifies the complete two-stage OTA after hierarchical reuse of the one-stage macromodel. Selected design points are translated back into circuit-level parameters and simulated with ngspice to compare the symbolic exploration results against transistor-level behavior.

In [124]:
simulation_wcos = OTA_2stage_macro_df_pareto[Symbol("W_cos")].to_numpy()
simulation_wdiff = OTA_2stage_macro_df_pareto[Symbol("W_diff")].to_numpy()
simulation_wal = OTA_2stage_macro_df_pareto[Symbol("W_al")].to_numpy()

wcos_max = 5e-6
ng_cos = np.ceil(simulation_wcos / wcos_max).astype(int)
ng_diff = np.ceil(simulation_wdiff / wcos_max).astype(int)
ng_al = np.ceil(simulation_wal / wcos_max).astype(int)

print("ng_cos", ng_cos)
print("ng_cos", ng_diff)
print("ng_cos", ng_al)

point = {
    Symbol("W_diff"): OTA_2stage_macro_df_pareto[Symbol("W_diff")].to_numpy(),
    Symbol("L_diff"): OTA_2stage_macro_df_pareto[Symbol("L_diff")].to_numpy(),
    Symbol("ng_diff"): ng_diff,
    Symbol("W_al"): OTA_2stage_macro_df_pareto[Symbol("W_al")].to_numpy(),
    Symbol("L_al"): OTA_2stage_macro_df_pareto[Symbol("L_al")].to_numpy(),
    Symbol("ng_al"): ng_al,
    Symbol("W_cos"): OTA_2stage_macro_df_pareto[Symbol("W_cos")].to_numpy(),
    Symbol("L_cos"): OTA_2stage_macro_df_pareto[Symbol("L_cos")].to_numpy(),
    Symbol("ng_cos"): ng_cos,
    Symbol("W_cs_1stage_m1"): OTA_2stage_macro_df_pareto[Symbol("W_cs_1stage_m1")].to_numpy(),
    Symbol("W_cs_1stage_m2"): OTA_2stage_macro_df_pareto[Symbol("W_cs_1stage_m2")].to_numpy(),
    Symbol("L_cs_1stage"): OTA_2stage_macro_df_pareto[Symbol("L_cs_1stage")].to_numpy(),
    Symbol("W_cs_2stage_m1"): OTA_2stage_macro_df_pareto[Symbol("W_cs_2stage_m1")].to_numpy(),
    Symbol("W_cs_2stage_m2"): OTA_2stage_macro_df_pareto[Symbol("W_cs_2stage_m2")].to_numpy(),
    Symbol("L_cs_2stage"): OTA_2stage_macro_df_pareto[Symbol("L_cs_2stage")].to_numpy(),
}

simulations = OTA_2stage_macro.ngspice_sim(
    point,
    variables=["gain", "vout"],
    extra_spice = {
        "pre": [
            "**",
            "x1 vref vret vout vbias_1stage vbias_2stage vdd vss OTA_2stage_macro", # VINP VINN VOUTP VDD IBIAS
            "R1 vfb vout 200000000 m=1",
            "C2 vfb vss 10 m=1",
            "R2 vfb vss 900000000 m=1",
            "V1 vret vfb dc 0 ac 1",
            "I0 vdd vbias_1stage 20e-6",
            "I1 vdd vbias_2stage 20e-6",
            ".lib /content/SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/models/cornerMOSlv.lib mos_tt",
            "Vref vref 0 0.9",
            "Vdd vdd 0 1.5",
            "Vss vss 0 0",
            ".control",
            "pre_osdi /content/SSTADEx/IHP-Open-PDK/ihp-sg13g2/libs.tech/ngspice/osdi/psp103_nqs.osdi",
            "ac dec 10 1 1G",
            "meas ac gain find vdb(vout) at=1000",
            "op",
            "print v(vout)",
            ".endc",
            ".end"
        ]
    })

print(simulations)

sim_results_df = pd.DataFrame(
    [
        {
            **sim_result["params"],
            **{
                f"sim_{name}": value
                for name, value in sim_result.get("variables", {}).items()
            },
            "run_name": sim_result["run_name"],
            "returncode": sim_result["returncode"],
        }
        for sim_result in simulations
    ]
)

ota_2stage_comparison_df = pd.concat(
    [
        OTA_2stage_macro_df_pareto.reset_index(drop=True),
        sim_results_df[
            ["run_name", "returncode", "sim_gain", "sim_vout"]
        ].reset_index(drop=True),
    ],
    axis=1,
)

ota_2stage_comparison_df["gain_error"] = 100*np.abs(ota_2stage_comparison_df["gain"] - ota_2stage_comparison_df["sim_gain"])/ota_2stage_comparison_df["gain"]
ota_2stage_comparison_df

## 6. Conclusion

This notebook presented SSTADEX as a structured and systematic Python framework for analog design exploration, moving from transistor-level LUT characterization to reusable primitives and hierarchical macromodel-based circuit design. Compared with our previous work, the implementation shown here provides a more practical and reusable software realization of the methodology, with clearer support for hierarchy construction, condition propagation, shared-node consistency, and result refinement across abstraction levels.

Through the one-stage and two-stage OTA examples, the notebook demonstrated how device-level data can be progressively transformed into higher-level design knowledge. The one-stage OTA served to illustrate the complete flow from primitive construction to symbolic exploration and circuit-level verification, while the two-stage OTA showed that previously developed blocks can be encapsulated as reusable macromodels and integrated into larger designs without redefining their internal structure.

Overall, the results support the viability of the SSTADEX approach for transparent, reusable, and hierarchically scalable analog design exploration. The verification results indicate that the explored abstractions can preserve good agreement with transistor-level simulation while significantly reducing the need for purely manual trial-and-error. This makes the framework especially useful for open, educational, and research-oriented analog design workflows, where reproducibility and interpretability are essential.